## EDA: Exploratory Data Analysis, is the way we can further review all the raw data from the dataset and check the null values and the valid ones. 

### ** We review the dataset making basic queries to check what we have**

In [0]:
%sql
/* Exploratory query */

SELECT * FROM bootcamp.raw.propiedades_bronze
LIMIT 10

### /* Identify column values */


In [0]:
%sql

DESCRIBE bootcamp.raw.propiedades_bronze




###  _NOTE_: We can see here that precio, numero, expensas, ambientes, metros_cuadrados_totales, metros_cuadrados_cubiertos, piso, antiguedad, all of them are string values, we need to think about it since perhaps it is necessary to make some insights to convert them to numeric values. We will do that later if it is necessary

### /* Dataset total records */


In [0]:
%sql


SELECT COUNT(*) FROM bootcamp.raw.propiedades_bronze

### Null values: 
Identify null values on the dataset, this null values are the most sensible to get important insights

In [0]:
%sql

SELECT
  COUNT(*) - COUNT(id) AS id_null,
  COUNT(*) - COUNT(ubicacion) AS ubicacion_null,
  COUNT(*) - COUNT(precio) AS precio_null,
  COUNT(*) - COUNT(zona) AS zona_null,
  COUNT(*) - COUNT(expensas) AS expensas_null,
  COUNT(*) - COUNT(moneda) AS moneda_null,
  COUNT(*) - COUNT(ambientes) AS ambientes_null,
  COUNT(*) - COUNT(tipo_de_operacion) AS tipo_de_operacion_null,
  COUNT(*) - COUNT(metros_cuadrados_totales) AS metros_cuadrados_totales_null,
  COUNT(*) - COUNT(metros_cuadrados_cubiertos) AS metros_cuadrados_cubiertos_null,
  COUNT(*) - COUNT(cochera) AS cochera_null,
  COUNT(*) - COUNT(antiguedad) AS antiguedad_null
FROM bootcamp.raw.propiedades_bronze

### Relation between null values and total records

Calculate the null values percentage

In [0]:
%sql

WITH total_records AS (
  SELECT COUNT(*) AS total FROM bootcamp.raw.propiedades_bronze
)
SELECT 
  ROUND((t.total - COUNT(ubicacion)) * 100 /  t.total, 2) AS ubicacion_null_pct, 
  ROUND((t.total - COUNT(precio)) * 100 /  t.total, 2) AS precio_null_pct, 
  ROUND((t.total - COUNT(expensas)) * 100 /  t.total, 2) AS expensas_null_pct, 
  ROUND((t.total - COUNT(moneda)) * 100 /  t.total, 2) AS moneda_null_pct, 
  ROUND((t.total - COUNT(ambientes)) * 100 /  t.total, 2) AS ambientes_null_pct, 
  ROUND((t.total - COUNT(tipo_de_operacion)) * 100 /  t.total, 2) AS tipo_de_operacion_null_pct, 
  ROUND((t.total - COUNT(metros_cuadrados_totales)) * 100 /  t.total, 2) AS metros_cuadrados_totales_null_pct, 
  ROUND((t.total - COUNT(metros_cuadrados_cubiertos)) * 100 /  t.total, 2) AS metros_cuadrados_cubiertos_null_pct,
  ROUND((t.total - COUNT(cochera)) * 100 /  t.total, 2) AS cochera_null_pct, 
  ROUND((t.total - COUNT(antiguedad)) * 100 /  t.total, 2) AS antiguedad_null_pct
FROM bootcamp.raw.propiedades_bronze
CROSS JOIN total_records t
GROUP BY t.total

From this query we can conclude:   
1 - Expensas percentage is too high with around 77% null values,   
2 - Same for cochera with around 80% null values  
Then, these fields probably will not be taken into consideration for more analysis.



### Once we know the null values, we review what type of them has the dataset. We know they are strings but we need to know the type of values are filled in the columns

In [0]:
%sql
--============================================================
--      MONEDA
--============================================================
SELECT 
  moneda,
  COUNT(*) AS total_records
FROM bootcamp.raw.propiedades_bronze
GROUP BY moneda
ORDER BY total_records DESC

In [0]:
%sql
--============================================================
--      TIPO DE OPERACION
--============================================================
SELECT
  tipo_de_operacion,
  COUNT(*) AS total_records,
  ROUND(COUNT(*) * 100 / SUM(COUNT(*)) OVER(), 2) as op_percentage
FROM bootcamp.raw.propiedades_bronze
GROUP BY tipo_de_operacion
ORDER BY total_records DESC

In [0]:
%sql
--============================================================
--      ANTIGUEDAD
--============================================================

SELECT 
  antiguedad,
  COUNT(*) AS total_records,
  ROUND(COUNT(*) * 100 / SUM(COUNT(*)) OVER(), 2) as ant_percentage
FROM bootcamp.raw.propiedades_bronze
GROUP BY antiguedad
ORDER BY total_records DESC

### NOTE: We found a placeholder '999' for ambientes, the way to handle the data for this project will be the same, we will keep this value understanding this is a null value for silver layer

In [0]:
%sql
-- ============================================================
--                  AMBIENTES
-- ============================================================

SELECT 
  ambientes,
  COUNT(*) AS total_records,
  ROUND(COUNT(*) * 100 / SUM(COUNT(*)) OVER(), 2) as amb_percentage
FROM bootcamp.raw.propiedades_bronze
GROUP BY ambientes
ORDER BY total_records DESC

### In this case the ambientes field needs to be reviewed, we have some info that does not have any sense.  

In [0]:
%sql
-- ============================================================
--                  ZONAS
-- ============================================================

SELECT 
  zona,
  COUNT(*) AS total_records,
  ROUND(COUNT(*) * 100 / SUM(COUNT(*)) OVER(), 2) AS zona_percentage
FROM bootcamp.raw.propiedades_bronze
GROUP BY zona
ORDER BY total_records DESC

-- IMPORTANT: we can see duplicated values here but with different names i.e. 'tigre' and 'gba-zona-norte--tigre', both are referred to the same neighbourhood

### At this point the dataset is with duplicated data for zonas, and for make a further analysis with metros and metros_cuadrados is imperative to convert these in decimal/float, since our original dataset is in string, we will proceed to convert all this values.

_Also, we will convert data and put it on a new temporary table to finish the EDA with the insigths related to decimal values, among with that we will join the info with other relevant columns_

In [0]:
%sql
-- Creating the temporary view
CREATE OR REPLACE TEMPORARY VIEW convert_values AS
SELECT 
  CASE 
    WHEN precio RLIKE '^[^a-zA-Z]+$' -- Why RLIKE? RLIKE is for Regular Expression, the regex '^[^a-zA-Z]+$' ensure the string is non-alphabetic
    THEN precio::double --  ::double convert the extracted sring to a double that allow us to calculate differents values
    ELSE null
  END AS conv_precio,
  tipo_de_operacion,
  moneda,
  CASE  
    WHEN ambientes RLIKE '^[^a-zA-Z]+$'
    THEN ambientes::double
    ELSE null
  END AS conv_ambientes,
  CASE 
    WHEN metros_cuadrados_totales RLIKE '^[^a-zA-Z]+$'
    THEN metros_cuadrados_totales::double
    ELSE null
  END AS conv_metros_totales,
  CASE 
    WHEN metros_cuadrados_cubiertos RLIKE '^[^a-zA-Z]+$'
    THEN metros_cuadrados_cubiertos::double
    ELSE null
  END AS conv_metros_cubiertos,
  CASE
    WHEN antiguedad RLIKE '^[^a-zA-Z]+$'
    THEN antiguedad::double
    ELSE null
  END AS conv_antiguedad,
  zona
FROM bootcamp.raw.propiedades_bronze

# THE TEMP VIEW HAVE TO RUN EVERY TIME THAT THE SERVERLESS DISCONNECTS

In [0]:
%sql
-- Checking the temp view
SELECT * FROM convert_values ORDER BY conv_metros_totales DESC LIMIT 10

### CURRENCY STATISTICS

In [0]:
%sql
SELECT
  moneda,
  COUNT(*) AS total_records,
  ROUND(MIN(conv_precio), 2) AS min_value,
  ROUND(MAX(conv_precio), 2) AS max_value,
  ROUND(AVG(conv_precio), 2) AS avg_value,
  ROUND(PERCENTILE(conv_precio, 0.5), 2) percentile_value
FROM convert_values
GROUP BY moneda
ORDER BY total_records DESC

### M2 and covered square meters

In [0]:
%sql
-- Data normalization: with UNION ALL transforms the result into a long format table

SELECT 
  'conv_metros_totales' AS type,
  ROUND(MIN(conv_metros_totales)) AS min_m2,
  ROUND(MAX(conv_metros_totales)) AS max_m2,
  ROUND(AVG(conv_metros_totales)) AS avg_m2,
  ROUND(PERCENTILE (conv_metros_totales, 0.5)) AS percentile_m2
FROM convert_values
WHERE conv_metros_totales > 0

UNION ALL 

SELECT 
  'conv_metros_cubiertos' AS type,
  ROUND(MIN(conv_metros_cubiertos)) AS min_m2,
  ROUND(MAX(conv_metros_cubiertos)) AS max_m2,
  ROUND(AVG(conv_metros_cubiertos)) AS avg_m2,
  ROUND(PERCENTILE (conv_metros_cubiertos, 0.5)) AS percentile_m2
FROM convert_values
WHERE conv_metros_cubiertos > 0


### PROPERTY AGE

In [0]:
%sql

-- Identyfing the outliers in the 'antiguedad' column

SELECT 
  conv_antiguedad,
  COUNT(*) AS total_records
FROM convert_values
GROUP BY conv_antiguedad
ORDER BY total_records DESC

999 value is a placeholder, then we calculate this value on the entire dataset to review total amount

In [0]:
%sql
--Calculate the 999 percentage

WITH temp AS (
  SELECT 
    COUNT(*) AS total_records
  FROM convert_values
)
SELECT 
  conv_antiguedad,
  ROUND(COUNT(conv_antiguedad) * 100 / MAX(total_records), 5) AS percentage
FROM convert_values
CROSS JOIN temp
GROUP BY conv_antiguedad
ORDER BY conv_antiguedad DESC


/*  ANOTHER WAY TO FIND IT
SELECT 
  conv_antiguedad,
  ROUND(COUNT(*) * 100 / SUM(COUNT(*)) OVER(), 5) AS percentage
FROM convert_values
GROUP BY conv_antiguedad
ORDER BY conv_antiguedad DESC
*/

This step shows us the 999 value is almost 99% dataset's value

In [0]:
%sql
-- Excluding the placeholder this is the insight we can get from the 'antiguedad' column
SELECT     
  COUNT(*) AS total_records,
  MIN(conv_antiguedad) AS min_value,
  MAX(conv_antiguedad) AS max_value,
  AVG(conv_antiguedad) AS avg_value,
  PERCENTILE(conv_antiguedad, 0.5) AS percentile_value
FROM convert_values
WHERE conv_antiguedad != 999
AND conv_antiguedad IS NOT NULL
AND conv_antiguedad > 0


### Quality Data

With the query below we review the quality data, this is the amount of null values and the percentage we have

In [0]:
%sql

WITH total_records AS (
  SELECT
    COUNT(*) AS total_recs
  FROM convert_values
),
  total_nulls AS (
SELECT 
  COUNT(CASE WHEN conv_precio IS NULL OR conv_precio <= 0 THEN 1 END) AS total_precio_null,
  COUNT(CASE WHEN conv_metros_totales IS NULL OR conv_metros_totales <= 0 THEN 1 END) AS total_metros_cubiertos_null,
  COUNT(CASE WHEN conv_antiguedad = 999 THEN 1 END) AS total_antiguedad_null,
  COUNT(CASE WHEN conv_ambientes IS NULL OR conv_ambientes <= 0 THEN 1 END) AS total_ambientes_null,
  COUNT(CASE WHEN moneda IS NULL THEN 1 END) AS total_moneda_null,
  COUNT(CASE WHEN tipo_de_operacion IS NULL THEN 1 END) AS total_tipo_de_operacion_null
FROM convert_values
)
SELECT 
  t.total_recs,
  tn.*,
  ROUND((total_precio_null * 100 / total_recs), 2) AS precio_null_percentage,
  ROUND((total_metros_cubiertos_null * 100 / total_recs), 2) AS metros_null_percentage,
  ROUND((total_antiguedad_null * 100 / total_recs), 2) AS antiguedad_null_percentage,
  ROUND((total_ambientes_null * 100 / total_recs), 2) AS ambientes_null_percentage,
  ROUND((total_moneda_null * 100 / total_recs), 2) AS moneda_null_percentage,
  ROUND((total_tipo_de_operacion_null * 100 / total_recs), 2) AS tipo_de_operacion_null_percentage
FROM total_records t,
total_nulls tn


### Duplicated records

In [0]:
%sql 

SELECT 
  precio,
  url,
  COUNT(*) AS duplicated_recs,
  COUNT(*) - 1 AS extra_records
FROM bootcamp.raw.propiedades_bronze
GROUP BY precio, url
HAVING COUNT(*) > 1
ORDER BY extra_records DESC
LIMIT 10


### OUTLIERS in precio

In [0]:
%sql

WITH percentile_values AS (
SELECT 
  moneda,
  ROUND(PERCENTILE(conv_precio, 0.01)) AS min_percentile,
  ROUND(PERCENTILE(conv_precio, 0.99)) AS max_percentile
FROM convert_values
WHERE conv_precio > 0
GROUP BY moneda
)
SELECT 
  cv.*,
  pv.*
FROM convert_values cv, percentile_values pv
WHERE cv.conv_precio < pv.min_percentile OR cv.conv_precio > pv.max_percentile

### Now we identified the apartment we count them to know the total amount outside of percentiles

In [0]:
%sql

WITH percentile_values AS (
  SELECT 
  moneda,
  ROUND(PERCENTILE(conv_precio, 0.01)) AS min_percentile,
  ROUND(PERCENTILE(conv_precio, 0.99)) AS max_percentile
FROM convert_values
WHERE conv_precio > 0
GROUP BY moneda
)
SELECT
  pv.moneda,
  'Under Min Percentile' AS under_min_prcntl,
  COUNT(*) AS total_min_prctl
FROM convert_values cv, percentile_values pv
WHERE cv.conv_precio < pv.min_percentile
GROUP BY pv.moneda

UNION ALL

SELECT
  pv.moneda,
  'Above Max Percentile' AS above_max_prcntl,
  COUNT(*) AS total_max_prctl
FROM convert_values cv, percentile_values pv
WHERE cv.conv_precio > pv.max_percentile
GROUP BY pv.moneda

### After all these steps we can make the final EDA to reach the final insights.  
### To do this we'll create a temporary view with all the columns taking into consideration to convert accordingly the corresponding values 

## DATA STANDARIZATION

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW final_EDA AS (
    SELECT
        pb.zona,
        pb.ubicacion,
        CASE
        /*          For precio, first at all, the column is filtered to nullify every amount that is not inside the values expected     */
            WHEN pb.precio = 'NaN' OR pb.precio NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL 
        /*          Furthermore, it is needed to check the digits to avoid an arithmetic overflow, for that reason the values in the between are the below  */
            WHEN pb.precio::float BETWEEN -2147483648 AND 2147483648 THEN precio::float -- Cap the range of values to avoid overflow, 32 bits integer range
            ELSE precio::float
        END AS precio,
        /*          About moneda, we transform the data to standard values and clean the currencies that are not valid     */ 
        CASE 
            WHEN LOWER(pb.moneda) LIKE '%usd%' THEN 'USD'
            WHEN LOWER(pb.moneda) LIKE '%dol%' THEN 'USD'
            WHEN LOWER(pb.moneda) LIKE '%u$%' THEN 'USD'
            WHEN LOWER(pb.moneda) LIKE '%dól%' THEN 'USD'
            WHEN LOWER(pb.moneda) LIKE '%peso%' THEN 'ARS'
            WHEN LOWER(pb.moneda) LIKE '%pesos%' THEN 'ARS'
            WHEN LOWER(pb.moneda) LIKE '%ar%' THEN 'ARS'
            WHEN LOWER(pb.moneda) LIKE '%ars%' THEN 'ARS'
            WHEN LOWER(pb.moneda) LIKE '% $%' THEN 'ARS'
        ELSE NULL
        END AS moneda,
        pb.calle,
        CASE
            WHEN pb.numero = 'NaN' OR pb.numero NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
            WHEN pb.numero::float BETWEEN -50000 AND 50000 THEN pb.numero::float
            ELSE NULL
        END AS numero,
        CASE
            WHEN pb.expensas = 'NaN' OR pb.expensas NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
            WHEN pb.expensas::float BETWEEN -2000000 AND 2000000 THEN pb.expensas::float
            ELSE NULL
        END AS expensas,
        pb.tipo_de_operacion,
        CASE
            WHEN pb.ambientes = 'NaN' OR pb.ambientes NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
            ELSE pb.ambientes::float
        END AS ambientes,
        pb.antiguedad,
        CASE 
            WHEN pb.metros_cuadrados_totales = 'NaN' OR pb.metros_cuadrados_totales NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
            ELSE pb.metros_cuadrados_totales::decimal
        END AS metros_cuadrados_totales,
        CASE 
            WHEN pb.metros_cuadrados_cubiertos = 'NaN' OR pb.metros_cuadrados_cubiertos NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
            ELSE pb.metros_cuadrados_cubiertos::decimal
        END AS metros_cuadrados_cubiertos,
        pb.orientacion_cardinal,
        pb.orientacion_inmueble,
        CASE
			when pb.piso IS NULL THEN NULL
			when pb.piso = 'NaN' then NULL
			else pb.piso::float
		END AS piso,
        pb.cochera,
        pb.estado,
        pb.tipo_vendedor,
        pb.original_text,
        pb.url,
		coalesce(try_cast(pb.fecha as date), current_date()) as fecha,        
        pb.hora,
        pb._rescued_data
    FROM bootcamp.raw.propiedades_bronze pb
)

In [0]:
%sql

SELECT * FROM final_EDA


In [0]:
%sql

WITH avg_zona AS (
SELECT 
  zona,
  moneda,
  tipo_de_operacion,
  ROUND(AVG(precio), 2) AS avg_precio,
  COUNT(*) AS total_recs
FROM final_EDA
WHERE moneda IS NOT NULL AND tipo_de_operacion = 'venta'
GROUP BY zona, moneda, tipo_de_operacion 
HAVING total_recs > 20
),
rank AS (
  SELECT ROW_NUMBER() OVER(PARTITION BY moneda ORDER BY avg_precio DESC) AS rank,
  zona,
  moneda,
  avg_precio,
  total_recs,
  tipo_de_operacion
FROM avg_zona
)
SELECT  
  rank,
  tipo_de_operacion,
  moneda,
  zona,
  avg_precio,
  total_recs
FROM rank
WHERE rank <= 10